# 记忆整理方式3：消息摘要，需要2个模型一个给agent使用，一个给摘要中间件使用


In [1]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model, SummarizationMiddleware
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from langchain_openai import ChatOpenAI

# 1.实例化模型
model_in = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)


# 1.实例化模型
model_out = ChatOpenAI(
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 2.创建agent对象
agent = create_agent(
    model=model_out,
    middleware=[SummarizationMiddleware(
        model=model_in,
        trigger=[
            ("tokens",100)
        ],
        keep=("messages",2), # 这个是最近的消息，必须保存完整的消息，不做摘要
        summary_prompt="对历史消息摘要，消息列表如下\n{messages}",
    )],
    checkpointer=InMemorySaver(),
)

config = {"configurable":{"thread_id":"1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]
for msg in conversations:
    resp = agent.invoke({"messages":[{"role":"user","content":msg}]},config=config)

    for msg1 in resp["messages"]:
        msg1.pretty_print()
    print("*"*50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三工程师！

我已经成功过滤掉那些“非常长非常长的废话”，准确接收到了你的核心信息：**你叫张三，是一名工程师。**

请问今天有什么我可以帮你的吗？无论是技术难题、代码调试，还是其他任何事情，尽管开口。
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to dat